In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
from datetime import date
from langdetect import detect, LangDetectException, DetectorFactory

DetectorFactory.seed = 0

In [ ]:
con = duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

silver_files = sorted(Path("../../data/silver").glob("cleaned_data_load_date=*.parquet"))
if not silver_files:
    raise FileNotFoundError("No Silver Parquet found. Run clean_transform_to_silver.ipynb first.")

silver_path = silver_files[-1]
print(f"Loading Silver from: {silver_path}")

silver_df = pd.read_parquet(silver_path)
print(f"Rows loaded:  {len(silver_df):,}")
print(f"Columns:      {list(silver_df.columns)}")

In [ ]:
def detect_language(text):
    """Return ISO 639-1 language code for text, or 'unknown' if detection fails or text is too short."""
    if text is None or str(text).strip() == "" or len(str(text).strip()) < 10:
        return "unknown"
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"

print("Running language detection on review_body ...")
silver_df["detected_language"] = silver_df["review_body"].apply(detect_language)

print("\nLanguage distribution (top 10):")
print(silver_df["detected_language"].value_counts().head(10))

In [ ]:
def type_token_ratio(text):
    """Lexical richness: unique_tokens / total_tokens. Returns None for empty/null text."""
    if text is None or str(text).strip() == "":
        return None
    tokens = str(text).lower().split()
    if len(tokens) == 0:
        return None
    return len(set(tokens)) / len(tokens)

silver_df["type_token_ratio"] = silver_df["review_body"].apply(type_token_ratio)

print("type_token_ratio distribution:")
print(silver_df["type_token_ratio"].describe().round(3))

In [ ]:
con.register("silver_with_lang", silver_df)

con.execute("DROP TABLE IF EXISTS gold.review_lexical_features")
con.execute("""
    CREATE TABLE gold.review_lexical_features AS
    WITH text_stats AS (
        SELECT
            _index,
            product_id,
            product_parent,
            product_title,
            vine,
            verified_purchase,
            review_headline,
            review_body,
            review_date,
            marketplace_id,
            product_category_id,
            label,
            record_origin,
            detected_language,
            type_token_ratio,

            -- Structural: word counts
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' '))     AS body_word_count,
            CASE
                WHEN review_headline IS NOT NULL AND TRIM(review_headline) != ''
                THEN ARRAY_LENGTH(STRING_SPLIT(TRIM(review_headline), ' '))
                ELSE 0
            END                                                     AS headline_word_count,

            -- Punctuation signal counts (char-subtraction method)
            (LENGTH(review_body) - LENGTH(REPLACE(review_body, '!', ''))) AS exclamation_count,
            (LENGTH(review_body) - LENGTH(REPLACE(review_body, '?', ''))) AS question_count,

            -- Sentence count approximation (periods as proxy, min 1 to avoid division-by-zero)
            GREATEST(
                LENGTH(review_body) - LENGTH(REPLACE(review_body, '.', '')), 1
            )                                                        AS sentence_count_approx,

            -- Average character length per word (total non-space chars / word count)
            (LENGTH(REPLACE(review_body, ' ', '')) * 1.0)
                / NULLIF(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')), 0)
                                                                     AS avg_word_length,

            -- Interaction terms encoded as categoricals for downstream ML
            -- vine_x_marketplace: Vine programme participation differs per market
            CONCAT(vine, '_', CAST(marketplace_id AS VARCHAR))       AS vine_x_marketplace,
            -- verified_x_category: purchase verification signal varies by product type
            CONCAT(
                verified_purchase, '_',
                CAST(product_category_id AS VARCHAR)
            )                                                        AS verified_x_category,

            -- HTML artifact density: HTML entities per word.
            -- High density signals copy-pasted web content rather than organic writing.
            -- Unique to this dataset — scraping preserved raw HTML that clean corpora lack.
            regexp_count(review_body, '&[a-zA-Z]+;|&#[0-9]+;') * 1.0
                / NULLIF(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')), 0)
                                                                     AS html_entity_density,

            -- Paragraph structure: <br> tag count proxies deliberate text organisation.
            -- Reviewers who paragraph their writing are typically more thorough.
            regexp_count(review_body, '<br\s*/?>')                   AS paragraph_break_count,
            CASE
                WHEN regexp_count(review_body, '<br\s*/?>') >= 2
                THEN TRUE ELSE FALSE
            END                                                      AS has_structured_body

        FROM silver_with_lang
        WHERE review_body IS NOT NULL
          AND TRIM(review_body) != ''
    ),
    lang_norms AS (
        SELECT
            *,
            -- Z-score of body_word_count within detected_language group.
            -- Removes cross-lingual verbosity bias so word count is comparable
            -- across French, German, and English reviews in a unified model.
            (body_word_count - AVG(body_word_count) OVER (PARTITION BY detected_language))
                / NULLIF(STDDEV(body_word_count) OVER (PARTITION BY detected_language), 0)
                AS body_lang_zscore,

            -- Category-level z-score: second, orthogonal normalisation axis.
            -- Book reviews are systematically longer than music reviews — this removes
            -- domain-level verbosity bias independently of language normalisation.
            (body_word_count - AVG(body_word_count) OVER (PARTITION BY product_category_id))
                / NULLIF(STDDEV(body_word_count) OVER (PARTITION BY product_category_id), 0)
                AS body_cat_zscore,

            -- Conciseness: how long the headline is relative to the body
            headline_word_count * 1.0
                / NULLIF(body_word_count, 0)                         AS headline_body_ratio,

            -- Punctuation densities per approximate sentence
            exclamation_count * 1.0
                / NULLIF(sentence_count_approx, 0)                   AS exclamation_density,
            question_count * 1.0
                / NULLIF(sentence_count_approx, 0)                   AS question_density

        FROM text_stats
    )
    SELECT * FROM lang_norms
""")

print("gold.review_lexical_features created.")

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                                                         AS total_rows,
        SUM(CASE WHEN detected_language = 'unknown' THEN 1 ELSE 0 END)  AS unknown_lang_count,
        SUM(CASE WHEN body_lang_zscore IS NULL THEN 1 ELSE 0 END)       AS null_lang_zscore,
        SUM(CASE WHEN body_cat_zscore IS NULL THEN 1 ELSE 0 END)        AS null_cat_zscore,
        SUM(CASE WHEN body_word_count < 1 THEN 1 ELSE 0 END)            AS zero_word_count,
        ROUND(AVG(body_word_count),      1)                              AS avg_body_words,
        ROUND(AVG(type_token_ratio),     3)                              AS avg_ttr,
        ROUND(AVG(avg_word_length),      2)                              AS avg_word_len,
        ROUND(AVG(html_entity_density),  4)                              AS avg_html_density,
        SUM(CASE WHEN has_structured_body THEN 1 ELSE 0 END)            AS structured_body_count
    FROM gold.review_lexical_features
""").df()

In [ ]:
con.execute("""
    SELECT
        detected_language,
        COUNT(*)                                  AS reviews,
        ROUND(AVG(body_word_count),     1)        AS avg_body_words,
        ROUND(AVG(body_lang_zscore),    3)        AS avg_zscore,
        ROUND(AVG(type_token_ratio),    3)        AS avg_ttr,
        ROUND(AVG(avg_word_length),     2)        AS avg_word_len,
        ROUND(AVG(exclamation_density), 3)        AS avg_excl_density,
        ROUND(AVG(question_density),    3)        AS avg_q_density
    FROM gold.review_lexical_features
    GROUP BY detected_language
    ORDER BY reviews DESC
""").df()

In [ ]:
con.execute("""
    SELECT
        label,
        COUNT(*)                                  AS reviews,
        ROUND(AVG(body_word_count),     1)        AS avg_body_words,
        ROUND(AVG(body_lang_zscore),    3)        AS avg_lang_zscore,
        ROUND(AVG(body_cat_zscore),     3)        AS avg_cat_zscore,
        ROUND(AVG(type_token_ratio),    3)        AS avg_ttr,
        ROUND(AVG(headline_body_ratio), 3)        AS avg_headline_ratio,
        ROUND(AVG(avg_word_length),     2)        AS avg_word_len,
        ROUND(AVG(exclamation_density), 4)        AS avg_excl_density,
        ROUND(AVG(question_density),    4)        AS avg_q_density,
        ROUND(AVG(html_entity_density), 4)        AS avg_html_density,
        ROUND(AVG(paragraph_break_count), 2)      AS avg_para_breaks,
        SUM(CASE WHEN has_structured_body THEN 1 ELSE 0 END) AS structured_count
    FROM gold.review_lexical_features
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label DESC
""").df()

In [ ]:
gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

output_file = gold_dir / f"lexical_features_load_date={date.today().isoformat()}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_lexical_features)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
""")
print(f"Saved: {output_file.resolve()}")

In [ ]:
con.close()